<a href="https://colab.research.google.com/github/Eligeti-13/Multimodal_AI_Projects/blob/asl/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install langchain==0.1.10

In [ ]:
!pip install pypdf sentence-transformers

In [ ]:
! pip install langchain_community ctransformers

In [ ]:
! pip install llama-cpp-python


In [ ]:
!pip install faiss-cpu

In [ ]:

!mkdir -p models

!wget -O models/llama-2-7b-chat.Q4_K_M.gguf \
https://huggingface.co/TheBloke/Llama-2-7B-Chat-GGUF/resolve/main/llama-2-7b-chat.Q4_K_M.gguf


In [7]:
!pip install -q streamlit

In [24]:
%%writefile RAGApp.py
import streamlit as st
import tempfile

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

from llama_cpp import Llama
from sentence_transformers import SentenceTransformer, util


st.set_page_config(page_title="Free RAG with LLaMA", layout="wide")

st.write("<h1 style='text-align: center;'>DocuRAG Intelligence</h1>", unsafe_allow_html=True)

st.sidebar.title("DocuFlow")
page = st.sidebar.radio(
    "Choose an option:",
    ["Dashboard", "Upload PDF", "Chat with PDF"])


@st.cache_resource
def load_embedding_model():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )

@st.cache_resource
def load_accuracy_model():
    return SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

@st.cache_resource
def load_llama_model():
    return Llama(model_path="models/llama-2-7b-chat.Q4_K_M.gguf")

embeddings = load_embedding_model()
accuracy_model = load_accuracy_model()
llm = load_llama_model()

# Dashboard
if page == "Dashboard":
    st.markdown("""DocuRAG Intelligence uses a Retrieval-Augmented Generation engine that
    fuses document search with AI reasoning. It extracts the right knowledge from your
    PDFs, injects it into your query, and produces answers that are evidence-driven,
    transparent, and reliable — not guesses.""")

    st.image("/content/RAG.png",width="stretch")

# Upload PDF
elif page == "Upload PDF":
    st.title("Upload PDF Document")
    uploaded_file = st.file_uploader("Upload a PDF", type=["pdf"])

    if uploaded_file:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(uploaded_file.read())
            pdf_path = tmp.name

        loader = PyPDFLoader(pdf_path)
        documents = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=200,
            chunk_overlap=50
        )

        chunks = splitter.split_documents(documents)

        vectorstore = FAISS.from_documents(chunks, embeddings)

        retriever = vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 2}
        )

        st.session_state["retriever"] = retriever
        st.success("PDF indexed successfully.")

# Chat with PDF
elif page == "Chat with PDF":
    st.title("Chat with Your PDF")

    if "retriever" not in st.session_state:
        st.warning("Please upload and index a PDF first.")
        st.stop()

    retriever = st.session_state["retriever"]

    query = st.text_input("Ask a question about the document")

    # Retrieve top-k chunks
    if query:
        docs = retriever.get_relevant_documents(query)
        context_text = "\n\n".join(doc.page_content for doc in docs)


        #prompt for LLaMA
        prompt = f"""
            You are a document-based question answering system.

            Rules:
            - Answer ONLY from the context below.
            - Do NOT use outside knowledge.
            - If the answer is not present in the context, say:
              "Not found in the document."

            Context:
            {context_text}

            Question:
            {query}

            Answer:
            """

        response = llm.create_completion(
            prompt=prompt,
            max_tokens=256,
            temperature=0
        )
        answer = response["choices"][0]["text"].strip()

        # Accuracy calculation
        answer_emb = accuracy_model.encode(answer, convert_to_tensor=True)
        context_emb = accuracy_model.encode(context_text, convert_to_tensor=True)
        accuracy = float(util.cos_sim(answer_emb, context_emb))

        # generating answer and accuracy
        st.subheader("Answer")
        st.write(answer)

        st.subheader("Accuracy")
        st.write(f"{accuracy:.2f}")



Overwriting RAGApp.py


In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


In [28]:
!nohup streamlit run RAGApp.py --server.port 8501 --server.address 0.0.0.0 >/dev/null 2>&1 &


In [ ]:
!./cloudflared tunnel --url http://localhost:8501 --no-autoupdate
